# Naive / baseline pocket-matching scores

Four reference baselines to compare against EPoCS, PocketVec, DrugCLIP and NRGRank on the PROSPECCTS benchmark:

1. **same_protein** - 1/0, are the two receptor chains the exact same full sequence (not just the same PDB code - many PDB entries are re-depositions, point mutants, or different complexes of the same protein).
2. **same_ligand** - 1/0, is the bound ligand the same chemical compound.
3. **pocket_seqid** - sequence identity of the pocket-lining residues, computed from a global alignment of the two full receptor sequences restricted to residues within 6 A of the ligand.
4. **ligand_tanimoto** - Morgan-fingerprint (radius 2, 2048 bit) Tanimoto similarity of the two bound ligands.

**Ligand identity problem.** The split `*_LIG.pdb` files (under `DATA/PROSPECCTS_split_pdbs/`) have every ligand HETATM residue renamed to a generic `LIG` - the real chemical identity isn't recoverable from those files alone. Most structure ids are `<pdbcode><chain>` (e.g. `1a42A` -> PDB `1a42`), so this notebook round-trips to RCSB: re-download the original deposited PDB, read the true chemical-component id at the same `(chain, resSeq)` (which survives the split), then fetch canonical SMILES for that id from the RCSB Chemical Component Dictionary. Two cases don't have a real, resolvable PDB code: `NMR_structures` (D2) uses synthetic ids (`cz00A`, `di00A`, ...), and ~83% of `decoy_structures` (D3/D4) ids are synthetic decoy placeholders (`A100A`, `A101A`, ...) never deposited to the PDB. For these - and for any other id where the round-trip fails for some other reason (e.g. the ligand was renumbered in the current RCSB deposition) - ligand identity falls back automatically to local, coordinate-based bond perception (RDKit `proximityBonding`); no internet needed there, but bond orders/stereochemistry are approximate. Where both sides of a pair have an authoritative RCSB chem-comp id, `same_ligand` uses exact id equality; otherwise it falls back to a near-exact (>=0.999) fingerprint match.

All network calls are cached to disk under `DATA/rcsb_cache/`, so re-running this notebook after the first pass is fast.

**Output** (drop-in for `02_benchmark_methods.ipynb`):
- `RESULTS/prospeccts_naive_baseline_dfs.pkl` - standalone `{dkey: DataFrame(p1, p2, lab, <4 score cols>)}`.
- `RESULTS/prospeccts_benchmark_dfs.pkl` - the canonical benchmark artifact, with the same 4 columns merged in alongside the existing method scores (pocketvec, nrgrank, drugclip variants, epocs).


In [1]:
import sys
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option('future.infer_string', False)  # keep plain object dtype for strings - see normalize_pair_dtypes below
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path.cwd()))
import naive_baselines_lib as nbl

RESULTS_DIR = nbl.RESULTS_DIR
BENCHMARK_PKL = RESULTS_DIR / 'prospeccts_benchmark_dfs.pkl'
BASELINE_PKL = RESULTS_DIR / 'prospeccts_naive_baseline_dfs.pkl'


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


def normalize_pair_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Force p1/p2/lab (and the column index itself) to plain object dtype
    before pickling - pandas >= 3.0 defaults to a StringDtype (for both data
    columns and the column Index) that older pandas can't unpickle."""
    df = df.copy()
    df.columns = pd.Index([str(c) for c in df.columns], dtype=object)
    for col in ('p1', 'p2', 'lab'):
        if col in df.columns:
            df[col] = df[col].astype(object)
    return df


out = load_pickle(BENCHMARK_PKL)
DKEYS = list(out.keys())
pd.DataFrame({k: len(out[k]) for k in DKEYS}, index=['n_pairs']).T


,n_pairs
D1,106276
D1.2,2025
D2,108241
D3,26860
D4,26860
D5,6400
D5.2,10000
D6,62
D7,56399


## Step 1 - locate structures, recover ligand identity

In [2]:
# Every dkey's pair list references ids that live under a specific
# PROSPECCTS_split_pdbs subdirectory (see naive_baselines_lib.DKEY_SPLIT_SUBDIR).
# Two different dkeys can point at the same directory (D5/D5.2 -> kahraman) or
# at genuinely different copies of a structure (D3 decoy_rational vs D4
# decoy_shape) - so everything below is cached per (split_dir, id), never per
# id alone.

dataset_ids = {}
for dkey in DKEYS:
    split_dir = nbl.SPLIT_PDBS_DIR / nbl.DKEY_SPLIT_SUBDIR[dkey]
    ids = sorted(set(out[dkey]['p1']) | set(out[dkey]['p2']))
    missing = [i for i in ids if not (split_dir / f'{i}_protein.pdb').is_file()]
    if missing:
        print(f'{dkey}: {len(missing)}/{len(ids)} ids missing protein.pdb under {split_dir} (showing 3): {missing[:3]}')
    dataset_ids[dkey] = (split_dir, ids)
    print(f'{dkey}: {len(ids)} unique ids -> {split_dir}')


D1: 326 unique ids -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/DATA/PROSPECCTS_split_pdbs/identical_structures/identical_structures
D1.2: 45 unique ids -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/DATA/PROSPECCTS_split_pdbs/identical_structures_similar_ligands/identical_structures_similar_ligands
D2: 329 unique ids -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/DATA/PROSPECCTS_split_pdbs/NMR_structures/NMR_structures
D3: 652 unique ids -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/DATA/PROSPECCTS_split_pdbs/decoy/decoy_rational_structures
D4: 652 unique ids -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/DATA/PROSPECCTS_split_pdbs/decoy/decoy_shape_structures
D5: 80 unique ids -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/DATA/PROSPECCTS_split_pdbs/kahraman_structures/kahraman_structures
D5.2: 100 unique ids -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/DATA/PROSPECCTS_split_pdbs/kahraman_structures/kahraman_structures
D6: 115 uni

In [3]:
# (split_dir, id) -> (chain, resseq, icode), ligand_atoms
lig_groups = {}
for dkey, (split_dir, ids) in dataset_ids.items():
    for pid in ids:
        key = (str(split_dir), pid)
        if key in lig_groups:
            continue
        lig_path = split_dir / f'{pid}_LIG.pdb'
        lig_key, lig_atoms = nbl.primary_ligand_group(lig_path)
        lig_groups[key] = (lig_key, lig_atoms)

print(f'{len(lig_groups)} unique (split_dir, id) structures total')

# RCSB round-trip only makes sense for ids that look like real PDB codes -
# NMR_structures (cz00A, di00A, ...) and ~83% of decoy_structures (A100A,
# A101A, ... - synthetic decoy placeholders) don't, and fall back to local
# bond perception automatically inside resolve_ligand_identity().
all_keys = sorted({
    (str(split_dir), pid)
    for dkey, (split_dir, ids) in dataset_ids.items()
    for pid in ids
})
pdb_codes_needed = {pid[:4] for (_, pid) in all_keys if nbl.looks_like_pdb_code(pid[:4])}
n_unresolvable = len(all_keys) - sum(1 for (_, pid) in all_keys if nbl.looks_like_pdb_code(pid[:4]))
print(f'{len(pdb_codes_needed)} unique PDB codes to fetch for ligand recovery '
      f'({n_unresolvable} ids are synthetic/non-PDB and go straight to the local fallback)')

t0 = time.time()
nbl.prefetch(pdb_codes_needed, nbl.fetch_original_pdb, max_workers=16, label='original PDB files')
print(f'  done in {time.time()-t0:.0f}s')


3370 unique (split_dir, id) structures total
1668 unique PDB codes to fetch for ligand recovery (981 ids are synthetic/non-PDB and go straight to the local fallback)
  prefetched 250/1668 original PDB files
  prefetched 500/1668 original PDB files
  prefetched 750/1668 original PDB files
  prefetched 1000/1668 original PDB files
  prefetched 1250/1668 original PDB files
  prefetched 1500/1668 original PDB files
  prefetched 1668/1668 original PDB files
  done in 0s


In [4]:
ligand_identity = {}  # (split_dir, id) -> dict(lig_code, fp)
for key in all_keys:
    split_dir_str, pid = key
    lig_key, lig_atoms = lig_groups[key]
    ligand_identity[key] = nbl.resolve_ligand_identity(pid[:4], lig_key, lig_atoms)

n_code = sum(1 for v in ligand_identity.values() if v['lig_code'])
n_fp = sum(1 for v in ligand_identity.values() if v['fp'] is not None)
print(f'authoritative chem-comp id recovered for {n_code}/{len(ligand_identity)} structures')
print(f'usable ligand fingerprint (RCSB SMILES or local fallback) for {n_fp}/{len(ligand_identity)} structures')

unique_lig_codes = {v['lig_code'] for v in ligand_identity.values() if v['lig_code']}
print(f'({len(unique_lig_codes)} unique ligand codes were resolved via RCSB CCD)')


[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerat

[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerat

[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] Can't kekulize mol.  Unkekulized atoms: 4 5 6 7 8 9 10 11 12
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] Can't kekulize mol.  Unkekulized atoms: 4 5 6 7 8 9 10 11 12
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING

[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:19] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerat

[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] DEPRECATION WARNING: please use MorganGenerator
[09:47:20] Can't kekulize mol.  Unkekulized at

...

[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerat

[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerat

[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] Can't kekulize mol.  Unkekulized atoms: 1 2 12 13 15
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGe

authoritative chem-comp id recovered for 2368/3370 structures
usable ligand fingerprint (RCSB SMILES or local fallback) for 3370/3370 structures
(877 unique ligand codes were resolved via RCSB CCD)


[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator
[09:47:35] DEPRECATION WARNING: please use MorganGenerator


## Step 2 - build per-structure cache: sequence, pocket residues

In [5]:
STRUCT = {}  # (split_dir, id) -> dict(seq, pocket_idx, lig_code, fp)

t0 = time.time()
n = 0
for dkey, (split_dir, ids) in dataset_ids.items():
    for pid in ids:
        key = (str(split_dir), pid)
        if key in STRUCT:
            continue
        protein_pdb = split_dir / f'{pid}_protein.pdb'
        seq, index_map, atoms_by_res = nbl.parse_protein_residues(protein_pdb)
        _lig_key, lig_atoms = lig_groups[key]
        pocket_keys = nbl.compute_pocket_residues(atoms_by_res, lig_atoms)
        pocket_idx = {index_map[k] for k in pocket_keys if k in index_map}

        lig_info = ligand_identity[key]
        STRUCT[key] = dict(seq=seq, pocket_idx=pocket_idx, lig_code=lig_info['lig_code'], fp=lig_info['fp'])
        n += 1
        if n % 500 == 0:
            print(f'  {n} structures cached ({time.time()-t0:.0f}s)')

print(f'built cache for {len(STRUCT)} structures in {time.time()-t0:.0f}s')
n_fp = sum(1 for v in STRUCT.values() if v['fp'] is not None)
print(f'ligand fingerprint available for {n_fp}/{len(STRUCT)} structures')


  500 structures cached (5s)


  1000 structures cached (10s)


  1500 structures cached (15s)


  2000 structures cached (20s)


  2500 structures cached (26s)


  3000 structures cached (32s)


built cache for 3370 structures in 36s
ligand fingerprint available for 3370/3370 structures


## Step 3 - pairwise scores

In [6]:
def pair_scores(sa, sb):
    same_protein = 1.0 if sa['seq'] == sb['seq'] else 0.0
    same_ligand = nbl.same_ligand_score(sa, sb)
    tan = nbl.tanimoto(sa['fp'], sb['fp'])
    pocket_seqid = nbl.pocket_sequence_identity(sa['seq'], sb['seq'], sa['pocket_idx'], sb['pocket_idx'])
    return same_protein, same_ligand, pocket_seqid, tan


t0 = time.time()
for dkey, (split_dir, ids) in dataset_ids.items():
    df = out[dkey]
    dir_str = str(split_dir)

    unique_pairs = {tuple(sorted((p1, p2))) for p1, p2 in zip(df['p1'], df['p2'])}
    pair_cache = {}
    for p1, p2 in unique_pairs:
        sa, sb = STRUCT[(dir_str, p1)], STRUCT[(dir_str, p2)]
        pair_cache[(p1, p2)] = pair_scores(sa, sb)

    same_protein_col, same_ligand_col, pocket_seqid_col, tanimoto_col = [], [], [], []
    for p1, p2 in zip(df['p1'], df['p2']):
        sp, sl, psid, tan = pair_cache[tuple(sorted((p1, p2)))]
        same_protein_col.append(sp)
        same_ligand_col.append(sl)
        pocket_seqid_col.append(psid)
        tanimoto_col.append(tan)

    df['same_protein_score'] = same_protein_col
    df['same_ligand_score'] = same_ligand_col
    df['pocket_seqid_score'] = pocket_seqid_col
    df['ligand_tanimoto_score'] = tanimoto_col

    print(f'{dkey}: {len(unique_pairs)} unique pairs scored ({len(df)} rows) - {time.time()-t0:.0f}s elapsed total')


D1: 53301 unique pairs scored (106276 rows) - 39s elapsed total


D1.2: 1035 unique pairs scored (2025 rows) - 40s elapsed total


D2: 54285 unique pairs scored (108241 rows) - 55s elapsed total


D3: 20308 unique pairs scored (26860 rows) - 61s elapsed total


D4: 20308 unique pairs scored (26860 rows) - 67s elapsed total


D5: 3240 unique pairs scored (6400 rows) - 71s elapsed total


D5.2: 5050 unique pairs scored (10000 rows) - 78s elapsed total
D6: 62 unique pairs scored (62 rows) - 78s elapsed total


D7: 55223 unique pairs scored (56399 rows) - 159s elapsed total


## Step 4 - save (standalone artifact + merged into the canonical benchmark pickle)

In [7]:
NEW_COLS = ['same_protein_score', 'same_ligand_score', 'pocket_seqid_score', 'ligand_tanimoto_score']

baseline_only = {
    dkey: normalize_pair_dtypes(out[dkey][['p1', 'p2', 'lab'] + NEW_COLS].copy())
    for dkey in DKEYS
}
save_pickle(baseline_only, BASELINE_PKL)
print(f'Saved standalone baselines -> {BASELINE_PKL}')

out = {k: normalize_pair_dtypes(v) for k, v in out.items()}
save_pickle(out, BENCHMARK_PKL)
print(f'Saved merged benchmark (existing methods + baselines) -> {BENCHMARK_PKL}')


Saved standalone baselines -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/RESULTS/prospeccts_naive_baseline_dfs.pkl
Saved merged benchmark (existing methods + baselines) -> /Users/jsutges/Documents/POCKET_MATCHING_BENCHMARK/RESULTS/prospeccts_benchmark_dfs.pkl


## Step 5 - sanity check: per-dataset AUC of each baseline

In [8]:
BASELINE_METHODS = [
    ('same_protein', 'same_protein_score'),
    ('same_ligand', 'same_ligand_score'),
    ('pocket_seqid', 'pocket_seqid_score'),
    ('ligand_tanimoto', 'ligand_tanimoto_score'),
]


def labels_to_int(s):
    return s.astype(str).str.lower().str.strip().map({'active': 1, 'inactive': 0})


def deduplicate_pairs(df):
    df = df[df['p1'] != df['p2']]
    key = df.apply(lambda r: frozenset((r['p1'], r['p2'])), axis=1)
    return df.loc[~key.duplicated()]


rows = []
for dkey in DKEYS:
    df = deduplicate_pairs(out[dkey])
    y = labels_to_int(df['lab'])
    row = {'dataset': dkey, 'n_pairs': len(df)}
    for label, col in BASELINE_METHODS:
        s = pd.to_numeric(df[col], errors='coerce')
        mask = y.notna() & s.notna()
        if mask.sum() == 0 or y[mask].nunique() < 2:
            row[label] = float('nan')
            continue
        row[label] = roc_auc_score(y[mask], s[mask])
    rows.append(row)

auc_table = pd.DataFrame(rows).set_index('dataset')
auc_table


,n_pairs,same_protein,same_ligand,pocket_seqid,ligand_tanimoto
dataset,,,,,
D1,52975,0.664606,0.499978,1.000000,0.784043
D1.2,990,0.867347,0.500000,1.000000,0.999794
D2,53956,1.000000,0.988649,1.000000,1.000000
D3,19982,0.664606,0.500000,1.000000,0.821186
D4,19982,0.664606,0.500000,1.000000,0.821659
D5,3160,0.500000,0.965476,0.533010,0.966540
D5.2,4950,0.500000,0.946721,0.518148,0.936064
D6,62,0.500000,0.470624,0.406977,0.460220
D7,55174,0.499964,0.600992,0.750881,0.737440


## Plugging into `02_benchmark_methods.ipynb`

To have the existing ROC / PR / threshold-sweep cells in `02_benchmark_methods.ipynb` pick these up automatically, extend its lookup tables (the pickle it loads already has the new columns after this notebook runs):

```python
METHODS += [
    ('same_protein',   'same_protein_score'),
    ('same_ligand',    'same_ligand_score'),
    ('pocket_seqid',   'pocket_seqid_score'),
    ('ligand_tanimoto','ligand_tanimoto_score'),
]
METHOD_COLORS.update({
    'same_protein':    'tab:gray',
    'same_ligand':     'tab:olive',
    'pocket_seqid':    'tab:cyan',
    'ligand_tanimoto': 'black',
})
METHOD_LABELS.update({
    'same_protein':    'Same protein (naive)',
    'same_ligand':     'Same ligand (naive)',
    'pocket_seqid':    'Pocket seq. identity',
    'ligand_tanimoto': 'Ligand Tanimoto',
})
```
